# 04 plugins

Question: do the natives of `golem-bridge` and the plugin runtime of `golem-plugin` do what
`okf/bridge.md`, `okf/plugins/index.md` and `okf/plugins/core.md` say, without a JVM.

The native methods of the host class convert their arguments and call one plain function of
`golem_bridge::host` each. `golem-devtool plugins` drives those functions with a script, as the
mod drives the natives on the server thread, so the notebook judges the same path. The notebook
holds the table of natives to the contract, then the order of the plugins, the cancel chain, the
panics, the budgets, the commands with the sources and levels of their senders, the permissions,
the core plugin, the wire form of the actions, the actions on players, the game with the guard of
the server thread and the wire form of its answers, the tools of operators, the queue, the tick
statistics and the pass values of a broken handler. The game is a scripted one in place of the
one the mod connects. The natives themselves run only inside a JVM, which the end to end test with
the Forge server and the bridge mod covers.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# The budget of one plugin call in the scripts, in milliseconds. A test plugin call takes
# microseconds, so a call over it is one that sleeps.
budget_ms = 20

# How long the slow test plugin sleeps, well over the budget.
slow_ms = 60

# The pause between the ticks of the wall clock section, in milliseconds.
tick_pause_ms = 100

## Setup

The cell finds the repository, builds `golem-devtool` and prints the versions of Python, cargo,
rustc and Golem. The helpers run a subcommand, write a script into a temporary directory and keep
the lines of the log that start with `[golem]`.

In [ ]:
import json
import re
import shutil
import sys
import tempfile
import tomllib
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(golem_root)
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-04-"))
WORKSPACE = tomllib.loads((P.root / "Cargo.toml").read_text(encoding="utf-8"))["workspace"]["package"]

run(["cargo", "build", "--quiet", "-p", "golem-devtool"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1", "--no-deps"], cwd=P.root, check=True).stdout)
DEVTOOL = Path(metadata["target_directory"]) / "debug" / "golem-devtool"
assert DEVTOOL.is_file(), "cargo built no golem-devtool"


def golem_lines(stderr):
    return [line[len("[golem] "):] for line in stderr.splitlines() if line.startswith("[golem] ")]


def devtool(*arguments, expect=0):
    result = run([DEVTOOL, *arguments], cwd=P.root, timeout=300)
    assert result.returncode == expect, "golem-devtool %s exited %d\n--- stdout\n%s--- stderr\n%s" % (
        " ".join(map(str, arguments)), result.returncode, result.stdout, result.stderr)
    return (json.loads(result.stdout) if result.stdout.strip() else None), golem_lines(result.stderr)


def write(name, document):
    path = SCRATCH / name
    path.write_text(json.dumps(document), encoding="utf-8")
    return path


def script(name, document):
    out, log = devtool("plugins", write(name + ".json", document))
    return out["steps"], log


def budgets_of(steps):
    rows = [step for step in steps if step["op"] == "budgets"][-1]["answer"]
    return {row["plugin"]: row for row in rows}


def broadcasts(drain_step):
    return [action["text"] for action in drain_step["decoded"] if action["kind"] == "broadcast"]


ALEX = {"uuid": "6f2d5b3e-1c4a-4e8b-9a7d-2b1c0e9f8a71", "name": "Alex"}
SAM = {"uuid": "0b9a8c7d-6e5f-4a3b-8c2d-1e0f9a8b7c6d", "name": "Sam"}
NETHER = {"dimension": "minecraft:the_nether", "x": 1.5, "y": 64.0, "z": -2.25, "yaw": 90.0, "pitch": -12.5}
NETHER_TEXT = "minecraft:the_nether 1.50 64.00 -2.25, yaw 90.0, pitch -12.5"
LAVA = {"dimension": "minecraft:overworld", "x": -10.0, "y": 11.0, "z": 300.75, "yaw": -45.0, "pitch": 0.0}
LAVA_TEXT = "minecraft:overworld -10.00 11.00 300.75, yaw -45.0, pitch 0.0"

natives_table, _ = devtool("natives")
print("python:", sys.version.split()[0])
print("cargo:", run(["cargo", "--version"], cwd=P.root, check=True).stdout.strip())
print("rustc:", run(["rustc", "--version"], cwd=P.root, check=True).stdout.strip())
print("golem:", natives_table["version"])
print("devtool:", P.rel(DEVTOOL) if P.rel(DEVTOOL) != str(DEVTOOL) else "golem-devtool in the target directory of cargo")

## The natives against the contract

`register_natives` hands the JVM a table of names, descriptors and functions. The cell reads the
host class of `okf/bridge.md`, derives the JNI descriptor of every native method from its Java
declaration and compares both lists name by name, in order. The version that `version()` answers
is the version of the workspace.

In [ ]:
bridge_md = (P.root / "okf" / "bridge.md").read_text(encoding="utf-8")
java = bridge_md.split("```java", 1)[1].split("```", 1)[0]
package = re.search(r"package ([\w.]+);", java).group(1)
klass = re.search(r"public final class (\w+)", java).group(1)
TYPES = {"String": "Ljava/lang/String;", "String[]": "[Ljava/lang/String;", "long": "J", "int": "I", "boolean": "Z",
         "double": "D", "float": "F", "void": "V", "Game": "L%s/Game;" % "com/skytekx/golem/host"}
declared = []
for returns, name, parameters in re.findall(r"public static native ([\w\[\]]+) (\w+)\(([^)]*)\);", java):
    types = [TYPES[" ".join(parameter.split()[:-1])] for parameter in parameters.split(",") if parameter.strip()]
    declared.append((name, "(%s)%s" % ("".join(types), TYPES[returns])))

registered = [(row["name"], row["signature"]) for row in natives_table["natives"]]
for (name, signature), (got_name, got_signature) in zip(declared, registered):
    print("%-15s %-24s %s" % (name, signature if len(signature) < 24 else signature[:21] + "...", "ok" if (name, signature) == (got_name, got_signature) else "WRONG"))
assert natives_table["hostClass"] == (package + "." + klass).replace(".", "/"), natives_table["hostClass"]
assert len(declared) == 16, "okf/bridge.md declares %d natives" % len(declared)
assert registered == declared, "the registered table differs from okf/bridge.md:\n%s\n%s" % (registered, declared)
assert natives_table["version"] == WORKSPACE["version"], natives_table["version"]
steps, _ = script("version", {"steps": [{"op": "version"}]})
assert steps[0]["answer"] == WORKSPACE["version"], steps

## The order of the plugins and the cancel chain

Two echo plugins broadcast the name of every event they see, and between them the plugin censor
cancels a chat message that holds the word spam and answers cancel to every other event. The
runtime must call the plugins in their order, end the chain at the first cancel of a chat
message, answer true for it, and ignore a cancel of an event that cannot be cancelled.

In [ ]:
steps, log = script("order", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "core"},
        {"kind": "echo", "name": "echo-a"},
        {"kind": "cancel", "name": "censor", "word": "spam"},
        {"kind": "echo", "name": "echo-b"},
    ],
    "steps": [
        {"op": "playerJoined", **ALEX},
        {"op": "drain"},
        {"op": "chat", **ALEX, "message": "hello"},
        {"op": "drain"},
        {"op": "chat", **ALEX, "message": "buy spam now"},
        {"op": "drain"},
        {"op": "respawn", **ALEX, "dimension": "minecraft:overworld"},
        {"op": "drain"},
        {"op": "budgets"},
    ],
})
assert steps[0]["answer"] is None, steps[0]
assert broadcasts(steps[1]) == ["echo-a playerJoined", "echo-b playerJoined"], steps[1]
assert steps[2]["answer"] is False, "a chat message without the word was cancelled"
assert broadcasts(steps[3]) == ["echo-a chat", "echo-b chat"], steps[3]
assert steps[4]["answer"] is True, "the chat message with the word was not cancelled"
assert broadcasts(steps[5]) == ["echo-a chat"], "the chain went on after the cancel: %s" % steps[5]
assert broadcasts(steps[7]) == ["echo-a respawn", "echo-b respawn"], steps[7]
calls = {name: row["calls"] for name, row in budgets_of(steps).items()}
print("calls:", calls)
assert calls == {"golem-core": 4, "echo-a": 4, "censor": 4, "echo-b": 3}, calls
assert log == [], log

## Panics are caught

The plugin fragile panics on every death and in its command. The runtime must catch each panic,
log it with the name of the plugin, count it, and go on as if the plugin passed, so the plugin
after it still sees the event with the place of the death. A command whose plugin panics answers
one line that says so.

In [ ]:
steps, log = script("panics", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "echo", "name": "echo-a"},
        {"kind": "panic", "name": "fragile", "on": "death", "commands": ["fragile"]},
        {"kind": "echo", "name": "echo-b"},
    ],
    "steps": [
        {"op": "death", **ALEX, "cause": "minecraft:fall", **NETHER},
        {"op": "death", **SAM, "cause": "minecraft:lava", **LAVA},
        {"op": "command", "name": "fragile", "senderUuid": ALEX["uuid"], "senderName": ALEX["name"], "arguments": "now"},
        {"op": "playerLeft", **ALEX},
        {"op": "drain"},
        {"op": "budgets"},
    ],
})
assert steps[0]["answer"] is None and steps[1]["answer"] is None, steps[:2]
assert steps[2]["answer"] == ["The command fragile failed, and the log of the server holds the cause."], steps[2]
assert broadcasts(steps[4]) == [
    "echo-a death at " + NETHER_TEXT, "echo-b death at " + NETHER_TEXT,
    "echo-a death at " + LAVA_TEXT, "echo-b death at " + LAVA_TEXT, "echo-a playerLeft", "echo-b playerLeft",
], steps[4]
fragile = budgets_of(steps)["fragile"]
print("fragile:", fragile)
assert fragile["calls"] == 4 and fragile["panics"] == 3, fragile
panics = [line for line in log if line.startswith("the plugin fragile panicked")]
for line in panics:
    print(line)
assert len(panics) == 3, log
assert sum("on the event death" in line for line in panics) == 2 and "on the command fragile" in panics[2], panics

## Budgets

The plugin sluggish sleeps longer than the budget on its first two joins. The runtime must time
every call, count both calls over the budget, keep the longest call, and log a call over the
budget at most once a minute per plugin. The budgets are printed as counts, since the times
differ from run to run. The status of the core plugin names the same counts.

In [ ]:
steps, log = script("budgets", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "core"},
        {"kind": "slow", "name": "sluggish", "on": "playerJoined", "sleepMs": slow_ms, "times": 2},
    ],
    "steps": [
        {"op": "playerJoined", **ALEX},
        {"op": "playerJoined", **SAM},
        {"op": "playerJoined", **ALEX},
        {"op": "tick", "nanos": 1000000, "repeat": 2},
        {"op": "budgets"},
        {"op": "command", "name": "golem", "arguments": "status"},
    ],
})
rows = budgets_of(steps)
print("budgets:", rows)
assert rows["sluggish"] == {"plugin": "sluggish", "calls": 5, "overBudget": 2, "panics": 0, "longestOverBudget": True}, rows
assert rows["golem-core"] == {"plugin": "golem-core", "calls": 5, "overBudget": 0, "panics": 0, "longestOverBudget": False}, rows
warnings = [line for line in log if "over the budget" in line]
print(warnings)
assert len(warnings) == 1 and warnings[0].startswith("the plugin sluggish took ") and "on the event playerJoined" in warnings[0], log
status = steps[5]["answer"]
for line in status:
    print(line)
assert status[1] == "Plugins: 2, with a budget of %.3f ms per call" % budget_ms, status
assert re.fullmatch(r"golem-core: calls 5, total [0-9.]+ ms, longest [0-9.]+ ms, over budget 0, panics 0", status[2]), status
assert re.fullmatch(r"sluggish: calls 5, total [0-9.]+ ms, longest [0-9.]+ ms, over budget 2, panics 0", status[3]), status
longest = float(re.search(r"longest ([0-9.]+) ms", status[3]).group(1))
assert longest >= slow_ms, "the longest call of sluggish took %.3f ms, less than its sleep" % longest

## Commands

The first plugin that declares a command owns it. A later plugin that declares the same name, or
a name that is empty or holds a space, is logged and refused. A command reaches its owner with its
sender, its level and its arguments, and an unknown command answers one line. A UUID and a name
are a player, neither at level 4 is the console, and a name alone is another source at its level,
such as a command block at level 2 or a zombie that runs the command at level 4. A UUID without a
name, neither of them below level 4 and a level outside 0 to 4 are refused rather than run as the
console, and a null `arguments` reads as the empty string.

In [ ]:
steps, log = script("commands", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "core"},
        {"kind": "echo", "name": "echo-a", "commands": ["echo-a"], "quiet": True},
        {"kind": "echo", "name": "echo-b", "commands": ["golem", "echo-a", "bad name", "", "echo-b"], "quiet": True},
    ],
    "steps": [
        {"op": "commandNames"},
        {"op": "command", "name": "echo-a", "senderUuid": ALEX["uuid"], "senderName": ALEX["name"], "level": 0,
         "arguments": "a b"},
        {"op": "command", "name": "echo-b", "level": 4, "arguments": ""},
        {"op": "command", "name": "echo-a"},
        {"op": "command", "name": "nope", "arguments": "x"},
        {"op": "command", "name": "echo-a", "senderName": "@", "level": 2, "arguments": "lamp"},
        {"op": "command", "name": "echo-b", "senderName": "Zombie", "level": 4},
        {"op": "command", "name": "echo-a", "senderUuid": ALEX["uuid"], "arguments": "x"},
        {"op": "command", "name": "echo-a", "level": 2, "arguments": "x"},
        {"op": "command", "name": "echo-a", "senderUuid": ALEX["uuid"], "senderName": ALEX["name"], "level": 5},
        {"op": "command", "name": "echo-a", "senderName": "@", "level": -1},
        {"op": "command", "arguments": "x"},
        {"op": "drain"},
        {"op": "budgets"},
    ],
})
assert steps[0]["answer"] == ["golem", "echo-a", "echo-b"], steps[0]
assert steps[1]["answer"] == ["echo-a ran echo-a for Alex with [a b]"], steps[1]
assert steps[2]["answer"] == ["echo-b ran echo-b for the console with []"], steps[2]
assert steps[3]["answer"] == ["echo-a ran echo-a for the console with []"], steps[3]
assert steps[4]["answer"] == ["The command nope is unknown."], steps[4]
assert steps[5]["answer"] == ["echo-a ran echo-a for @ at level 2 with [lamp]"], steps[5]
assert steps[6]["answer"] == ["echo-b ran echo-b for Zombie at level 4 with []"], steps[6]
assert all(step["answer"] == [] for step in steps[7:12]), steps[7:12]
assert steps[12]["decoded"] == [
    {"kind": "message", "uuid": ALEX["uuid"], "text": "echo-a heard Alex"},
    {"kind": "command", "command": "say echo-a ran echo-a"},
    {"kind": "command", "command": "say echo-b ran echo-b"},
    {"kind": "command", "command": "say echo-a ran echo-a"},
    {"kind": "command", "command": "say echo-a ran echo-a"},
    {"kind": "command", "command": "say echo-b ran echo-b"},
], steps[12]
assert budgets_of(steps)["echo-a"]["calls"] == 3 and budgets_of(steps)["echo-b"]["calls"] == 2, budgets_of(steps)
for line in log:
    print(line)
assert log == [
    "the plugin echo-b declares the command golem, which the plugin golem-core owns, so the runtime ignores it",
    "the plugin echo-b declares the command echo-a, which the plugin echo-a owns, so the runtime ignores it",
    "the plugin echo-b declares the command \"bad name\", which is no valid name, so the runtime ignores it",
    "the plugin echo-b declares the command \"\", which is no valid name, so the runtime ignores it",
    "the native command got senderUuid without senderName, so it refuses the command echo-a",
    "the native command got neither senderUuid nor senderName at the level 2, which only the console has at 4, so it refuses the command echo-a",
    "the native command got the level 5, which is no permission level, so it refuses the command echo-a",
    "the native command got the level -1, which is no permission level, so it refuses the command echo-a",
    "the native command got null for name, so it answers the pass value",
], log

## Permissions

The mod asks the native permission for every node of type Boolean of the PermissionAPI of Forge.
The first plugin that answers decides, a plugin that answers nothing leaves the node to the next,
a plugin that panics is caught and counted and the chain goes on, and without any answer the node
falls to its default resolver, which the native answers as none. A null node is logged and
answered as none.

In [ ]:
steps, log = script("permissions", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "core"},
        {"kind": "grant", "name": "ranks", "grants": ["forge.use_entity_selectors"], "denies": ["golem.fly"]},
        {"kind": "panic", "name": "fragile", "on": "permission"},
        {"kind": "grant", "name": "late", "grants": ["golem.fly", "golem.late"]},
    ],
    "steps": [
        {"op": "permission", "uuid": ALEX["uuid"], "node": "forge.use_entity_selectors"},
        {"op": "permission", "uuid": ALEX["uuid"], "node": "golem.fly"},
        {"op": "permission", "uuid": SAM["uuid"], "node": "golem.late"},
        {"op": "permission", "uuid": SAM["uuid"], "node": "golem.unknown"},
        {"op": "permission", "uuid": SAM["uuid"]},
        {"op": "budgets"},
    ],
})
answers = [step["answer"] for step in steps[:5]]
print("answers:", answers)
assert answers == [True, False, True, None, None], answers
calls = {name: (row["calls"], row["panics"]) for name, row in budgets_of(steps).items()}
print("calls and panics:", calls)
assert calls == {"golem-core": (4, 0), "ranks": (4, 0), "fragile": (2, 2), "late": (2, 0)}, calls
# The first panic of a process may take longer than the budget, which the log may name too.
log = [line for line in log if " over the budget of " not in line]
for line in log:
    print(line)
assert [line.split(" with ")[0] for line in log] == [
    "the plugin fragile panicked on the permission golem.late",
    "the plugin fragile panicked on the permission golem.unknown",
    "the native permission got null for node, so it answers the pass value",
], log

## The core plugin

`/golem source` names the license, the version, the repository and the commit, and links the tree
of that commit, as section 13 of the AGPL asks, or the repository alone while the commit is
unknown. `/golem status` names the TPS, the MSPT and the ticks of the bridge. `/golem` and
`/golem help` list the subcommands, and an unknown subcommand answers an error and the help. The
core plugin logs the connection of the bridge once, however often the server starts.

In [ ]:
REPOSITORY = WORKSPACE["repository"]
COMMIT = "4b825dc642cb6eb9a060e54bf8d69288fbee4904"
core = [{"kind": "core"}]
steps, log = script("core", {
    "budgetMs": budget_ms,
    "source": {"version": WORKSPACE["version"], "repository": REPOSITORY, "commit": COMMIT},
    "plugins": core,
    "steps": [
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
        {"op": "commandNames"},
        {"op": "command", "name": "golem", "arguments": "source"},
        {"op": "command", "name": "golem"},
        {"op": "command", "name": "golem", "arguments": "help"},
        {"op": "command", "name": "golem", "arguments": "frobnicate now"},
        {"op": "tick", "nanos": 4000000, "repeat": 3},
        {"op": "stats"},
        {"op": "command", "name": "golem", "arguments": "  STATUS  "},
    ],
})
assert log.count("the bridge is connected: Forge 47.4.18, Minecraft 1.20.1") == 1 and len(log) == 1, log
assert steps[2]["answer"] == ["golem"], steps[2]
source = steps[3]["answer"]
for line in source:
    print(line)
assert source[0] == "Golem %s is free software under the GNU Affero General Public License 3.0 only, AGPL-3.0-only." % WORKSPACE["version"], source
assert "Section 13 of the AGPL" in source[1], source
assert source[2:] == ["Repository: " + REPOSITORY, "Commit: " + COMMIT, "Source of this build: %s/tree/%s" % (REPOSITORY, COMMIT)], source
help_lines = steps[4]["answer"]
assert steps[5]["answer"] == help_lines and len(help_lines) == 9, help_lines
assert [line.split(":")[0] for line in help_lines[1:]] == [
    "/golem source", "/golem status", "/golem whoami", "/golem player <name>", "/golem world [<dimension>]",
    "/golem teleport <name> <dimension> <x> <y> <z>", "/golem listname <name> [<text>]", "/golem help"], help_lines
assert steps[6]["answer"] == ["The subcommand frobnicate of /golem is unknown."] + help_lines, steps[6]
stats = steps[8]["answer"]
assert steps[9]["answer"][0] == "Golem %s: TPS %.2f, MSPT %.2f, ticks %d" % (WORKSPACE["version"], stats["tps"], stats["mspt"], stats["ticks"]), (steps[9], stats)
assert stats["ticks"] == 3 and stats["mspt"] == 4.0, stats

unknown, _ = script("core-unknown", {
    "source": {"version": WORKSPACE["version"], "repository": REPOSITORY + "/", "commit": "unknown"},
    "plugins": core,
    "steps": [{"op": "command", "name": "golem", "arguments": "source"}],
})
assert unknown[0]["answer"][3:] == ["Commit: unknown", "Source of this build: " + REPOSITORY], unknown
build, _ = script("core-build", {
    "source": "build",
    "plugins": core,
    "steps": [{"op": "command", "name": "golem", "arguments": "source"}],
})
lines = build[0]["answer"]
commit = lines[3].removeprefix("Commit: ")
assert lines[2] == "Repository: " + REPOSITORY, lines
assert lines[4] == "Source of this build: " + (REPOSITORY if commit == "unknown" else "%s/tree/%s" % (REPOSITORY, commit)), lines
print("the build names the commit", commit)

## The wire form of the actions

Every action is one string, its fields joined by U+001F, the first field its kind. A field that
holds U+001F has it replaced by a space, and the first replacement of a process is logged, later
ones are not. A number of a teleport is the shortest decimal that reads back as the same number,
and an empty list name clears the name. A line of an unknown kind, with a wrong number of fields
or with a number that does not read decodes to nothing.

In [ ]:
US = "\x1f"
TELEPORT = {"kind": "teleport", "uuid": ALEX["uuid"], **NETHER}
entries = [
    {"kind": "command", "command": "save-all flush"},
    {"kind": "message", "uuid": ALEX["uuid"], "text": "Welcome back"},
    {"kind": "broadcast", "text": "The server restarts in 5 minutes."},
    TELEPORT,
    {"kind": "listName", "uuid": ALEX["uuid"], "name": "[AFK] Alex"},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": None},
    {"kind": "message", "uuid": ALEX["uuid"], "text": "a" + US + "b"},
    {"kind": "broadcast", "text": US + "x" + US},
    {"kind": "command", "command": "say " + US},
    {"kind": "teleport", "uuid": ALEX["uuid"], "dimension": "minecraft:overworld", "x": 0.1, "y": -64.0, "z": 1e-07,
     "yaw": 359.5, "pitch": -90.0},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": ""},
    {"wire": "explode" + US + ALEX["uuid"]},
    {"wire": "message" + US + ALEX["uuid"]},
    {"wire": "broadcast" + US + "a" + US + "b"},
    {"wire": "teleport" + US + ALEX["uuid"] + US + "minecraft:overworld" + US + "1" + US + "2" + US + "3" + US + "0"},
    {"wire": US.join(["teleport", ALEX["uuid"], "minecraft:overworld", "1", "two", "3", "0", "0"])},
    {"wire": "listName" + US + ALEX["uuid"]},
    {"wire": "broadcast" + US},
]
rows, log = devtool("actions-encode", write("actions.json", entries))
for row in rows:
    print(repr(row["wire"].replace(US, "|")), row["decoded"])
assert rows[0]["wire"] == "command" + US + "save-all flush", rows[0]
assert rows[1]["wire"] == "message" + US + ALEX["uuid"] + US + "Welcome back", rows[1]
assert rows[2]["wire"] == "broadcast" + US + "The server restarts in 5 minutes.", rows[2]
assert rows[3]["fields"] == ["teleport", ALEX["uuid"], "minecraft:the_nether", "1.5", "64", "-2.25", "90", "-12.5"], rows[3]
assert rows[4]["fields"] == ["listName", ALEX["uuid"], "[AFK] Alex"], rows[4]
assert rows[5]["fields"] == ["listName", ALEX["uuid"], ""], rows[5]
for row, entry in zip(rows[:6], entries[:6]):
    assert row["decoded"] == entry and row["reencoded"] == row["wire"], row
assert rows[6]["fields"] == ["message", ALEX["uuid"], "a b"], rows[6]
assert rows[7]["fields"] == ["broadcast", " x "], rows[7]
assert rows[8]["fields"] == ["command", "say  "], rows[8]
assert rows[9]["fields"][3:] == ["0.1", "-64", "0.0000001", "359.5", "-90"], rows[9]
assert rows[9]["decoded"] == entries[9], rows[9]
assert rows[10]["fields"] == rows[5]["fields"] and rows[10]["decoded"]["name"] is None, rows[10]
assert [row["decoded"] for row in rows[11:17]] == [None] * 6, rows[11:17]
assert rows[17]["decoded"] == {"kind": "broadcast", "text": ""}, rows[17]
assert log == ["a field of a message action holds U+001F, which the wire form reserves, so it is sent as a space, and later fields like it are not logged again"], log

## The actions on players

The plugin mover queues its actions through the helpers of the context. A teleport goes into the
queue with its place as it is, a place whose coordinate is no number is refused with one line on
the log and queues nothing, and a list name is set and cleared.

In [ ]:
steps, log = script("mover", {
    "plugins": [{"kind": "mover", "name": "mover", "commands": ["move"], "to": NETHER}],
    "steps": [
        {"op": "command", "name": "move", "arguments": "teleport " + ALEX["uuid"]},
        {"op": "command", "name": "move", "arguments": "nowhere " + ALEX["uuid"]},
        {"op": "command", "name": "move", "arguments": "name " + ALEX["uuid"] + " [AFK] Alex"},
        {"op": "command", "name": "move", "arguments": "clear " + ALEX["uuid"]},
        {"op": "drain"},
    ],
})
assert [step["answer"] for step in steps[:4]] == [["queued true"], ["queued false"], ["named"], ["cleared"]], steps
assert steps[4]["decoded"] == [
    TELEPORT,
    {"kind": "listName", "uuid": ALEX["uuid"], "name": "[AFK] Alex"},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": None},
], steps[4]
print(log)
assert log == ["a plugin asked to teleport %s to minecraft:the_nether NaN 64.00 -2.25, yaw 90.0, pitch -12.5, which is no finite "
               "place, so the runtime refuses it" % ALEX["uuid"]], log

## The game

A plugin asks the game through its context while a native of the server thread runs. The script
hands the natives a scripted game in place of the one the mod connects. The server thread is the
thread of the first `serverStarting`, so a question before it answers none, and the first such
question of the process is logged once. Then every question answers what the game holds, none
where the game holds none, and nothing for a player who is not online or a dimension the server
does not have. A safe place keeps the dimension and the rotation of its target, and a search
that meets a chunk that is not loaded answers so instead of a place. A chat message
reaches the plugins on a thread of its own, as Forge calls `chat` on a worker thread, and its
question answers none without a second line on the log, while a tick on the server thread gets
its answer. Without a game every question answers nothing.

In [ ]:
OVERWORLD = {"id": "minecraft:overworld", "spawn": {"x": 8, "y": 70, "z": -16, "angle": 90.0}, "minY": -64,
             "maxY": 320, "safe": {"x": 0.5, "y": 65.0, "z": 2.5}}
NETHER_DIMENSION = {"id": "minecraft:the_nether", "spawn": {"x": 8, "y": 70, "z": -16}, "minY": 0, "maxY": 256}
END_DIMENSION = {"id": "minecraft:the_end", "spawn": {"x": 100, "y": 50, "z": 0}, "minY": 0, "maxY": 256,
                 "safe": "unloaded"}
GAME = {
    "players": [
        {**ALEX, "displayName": "Alex the Brave", "ping": 12, "gameMode": "creative", "level": 2, "idleMs": 1500,
         "location": NETHER, "respawn": {"dimension": "minecraft:overworld", "x": 10, "y": 64, "z": -20},
         "mainHand": {"id": "minecraft:diamond", "count": 3}},
        {**SAM, "listName": "[AFK] Sam", "idleMs": 900000, "location": LAVA},
    ],
    "dimensions": [OVERWORLD, NETHER_DIMENSION, END_DIMENSION],
}
ASK = {"kind": "ask", "name": "ask", "commands": ["ask"], "on": "chat"}
questions = ["players", "player alex", "player " + SAM["uuid"].upper(), "player nobody", "location " + ALEX["uuid"],
             "respawn " + ALEX["uuid"], "respawn " + SAM["uuid"], "spawn minecraft:overworld", "spawn golem:void",
             "hand " + ALEX["uuid"], "hand " + SAM["uuid"], "heights minecraft:the_nether",
             "safe minecraft:overworld 0.2 64 2.9 3", "safe minecraft:the_nether 0 64 0 3", "safe golem:void 0 0 0 3",
             "safe minecraft:the_end 0 64 0 3", "location 00000000-0000-4000-8000-000000000000"]
steps, log = script("game", {
    "plugins": [ASK],
    "game": GAME,
    "steps": [{"op": "command", "name": "ask", "arguments": "players"},
              {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"}]
             + [{"op": "command", "name": "ask", "arguments": question} for question in questions]
             + [{"op": "together", "steps": [{"op": "chat", **SAM, "message": "anyone here"}]},
                {"op": "tick", "nanos": 1000000},
                {"op": "chat", **ALEX, "message": "on the server thread"},
                {"op": "drain"}],
})
before = json.loads(steps[0]["answer"][0])
answers = {question: json.loads(step["answer"][0]) for question, step in zip(questions, steps[2:])}
for question, value in answers.items():
    print("%-60s %s" % (question[:60], json.dumps(value)[:120]))
assert before is None, before
players = answers["players"]
assert [player["name"] for player in players] == ["Alex", "Sam"], players
assert players[0] == {"uuid": ALEX["uuid"], "name": "Alex", "displayName": "Alex the Brave", "listName": None,
                      "ping": 12, "gameMode": "creative", "level": 2, "idleMs": 1500}, players[0]
assert players[1]["listName"] == "[AFK] Sam" and players[1]["gameMode"] == "survival" and players[1]["level"] == 0
assert answers["player alex"] == players[0] and answers["player " + SAM["uuid"].upper()] == players[1], answers
assert answers["player nobody"] == {"none": True}, answers["player nobody"]
assert answers["location " + ALEX["uuid"]] == NETHER, answers
assert answers["respawn " + ALEX["uuid"]] == {"dimension": "minecraft:overworld", "x": 10, "y": 64, "z": -20}
assert answers["respawn " + SAM["uuid"]] == {"none": True}
assert answers["spawn minecraft:overworld"] == {"dimension": "minecraft:overworld", "x": 8, "y": 70, "z": -16,
                                                 "angle": 90.0}
assert answers["spawn golem:void"] is None
assert answers["hand " + ALEX["uuid"]] == {"id": "minecraft:diamond", "count": 3}
assert answers["hand " + SAM["uuid"]] == {"none": True}
assert answers["heights minecraft:the_nether"] == {"min": 0, "max": 256}
assert answers["safe minecraft:overworld 0.2 64 2.9 3"] == {"dimension": "minecraft:overworld", "x": 0.5, "y": 65.0,
                                                           "z": 2.5, "yaw": 0.0, "pitch": 0.0}
assert answers["safe minecraft:the_nether 0 64 0 3"] == {"none": True}
assert answers["safe golem:void 0 0 0 3"] is None
assert answers["safe minecraft:the_end 0 64 0 3"] == {"unloaded": True}, answers["safe minecraft:the_end 0 64 0 3"]
assert answers["location 00000000-0000-4000-8000-000000000000"] is None
chat_off_thread = steps[-4]["answer"][0]
assert chat_off_thread["answer"] is False, chat_off_thread
assert broadcasts(steps[-1]) == ["ask got no answer", "ask saw 2 players"], steps[-1]
for line in log:
    print(line)
assert log == ["a plugin asked the game for players off the server thread, so the game answers none, and later "
               "questions like it are not logged"], log

without, log = script("no-game", {
    "plugins": [ASK],
    "steps": [{"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
              {"op": "command", "name": "ask", "arguments": "players"},
              {"op": "command", "name": "ask", "arguments": "safe minecraft:overworld 0 64 0 3"}],
})
assert [json.loads(step["answer"][0]) for step in without[1:]] == [None, None], without
assert log == [], log

## The wire form of the game

The mod answers every question as fields joined by U+001F, with the numbers as Java writes them.
`game-wire` reads answers as the natives read them. A number in the exponent form of Java reads,
a coordinate that is no finite number, a game mode the game does not have, a level above 4 and a
wrong number of fields do not read, a ping below 0 counts as 0, an empty name in the list is no
name, and the empty answer of a respawn point, a hand and a safe place is none. A safe place
keeps the dimension and the rotation of its target, and the answer `unloaded` says that the
search met a chunk that is not loaded.

In [ ]:
def joined(*fields):
    return US.join(str(field) for field in fields)


PLAYER = [ALEX["uuid"], "Alex", "Alex", 12, "survival", 0, 1500, ""]
TARGET = {"dimension": "minecraft:the_nether", "x": 0.0, "y": 100.0, "z": 0.0, "yaw": 45.0, "pitch": 10.0}
entries = [
    {"question": "players", "answer": joined(*PLAYER)},
    {"question": "players", "answer": joined(*PLAYER[:7], "[AFK] Alex")},
    {"question": "players", "answer": joined(*PLAYER[:3], -3, *PLAYER[4:])},
    {"question": "players", "answer": joined(*PLAYER[:4], "hardcore", *PLAYER[5:])},
    {"question": "players", "answer": joined(*PLAYER[:5], 5, *PLAYER[6:])},
    {"question": "players", "answer": joined(*PLAYER[:7])},
    {"question": "location", "answer": joined("minecraft:overworld", 1.5, 64.0, -3.25, 90.0, -12.5)},
    {"question": "location", "answer": joined("minecraft:overworld", "1.0E-5", "-0.0", "3.0E10", "0.0", "0.0")},
    {"question": "location", "answer": joined("minecraft:overworld", "NaN", 64.0, 0.0, 0.0, 0.0)},
    {"question": "location", "answer": joined("minecraft:overworld", "Infinity", 64.0, 0.0, 0.0, 0.0)},
    {"question": "location", "answer": joined("minecraft:overworld", 1.5, 64.0, -3.25, 90.0)},
    {"question": "respawn", "answer": ""},
    {"question": "respawn", "answer": joined("minecraft:overworld", 10, 64, -20)},
    {"question": "respawn", "answer": joined("minecraft:overworld", "ten", 64, -20)},
    {"question": "spawn", "answer": joined(8, 70, -16, "90.0"), "dimension": "minecraft:overworld"},
    {"question": "mainHand", "answer": ""},
    {"question": "mainHand", "answer": joined("minecraft:diamond", 3)},
    {"question": "heights", "answer": joined(-64, 320)},
    {"question": "safeLocation", "answer": "", "target": TARGET},
    {"question": "safeLocation", "answer": joined("-0.5", "100.0", "0.5"), "target": TARGET},
    {"question": "safeLocation", "answer": "unloaded", "target": TARGET},
    {"question": "safeLocation", "answer": "loaded", "target": TARGET},
]
rows, log = devtool("game-wire", write("game-wire.json", entries))
decoded = [row["decoded"] for row in rows]
for row in rows:
    print("%-12s %-60r %s" % (row["question"], row["answer"].replace(US, "|")[:60], json.dumps(row["decoded"])[:80]))
expected_player = {"uuid": ALEX["uuid"], "name": "Alex", "displayName": "Alex", "listName": None, "ping": 12,
                   "gameMode": "survival", "level": 0, "idleMs": 1500}
assert decoded[0] == expected_player, decoded[0]
assert decoded[1] == {**expected_player, "listName": "[AFK] Alex"}, decoded[1]
assert decoded[2] == {**expected_player, "ping": 0}, decoded[2]
assert decoded[3:6] == [None, None, None], decoded[3:6]
assert decoded[6] == {"dimension": "minecraft:overworld", "x": 1.5, "y": 64.0, "z": -3.25, "yaw": 90.0,
                      "pitch": -12.5}, decoded[6]
assert decoded[7] == {"dimension": "minecraft:overworld", "x": 1e-05, "y": -0.0, "z": 3e10, "yaw": 0.0,
                      "pitch": 0.0}, decoded[7]
assert decoded[8:11] == [None, None, None], decoded[8:11]
assert decoded[11] == {"none": True} and decoded[13] is None, decoded[11:14]
assert decoded[12] == {"dimension": "minecraft:overworld", "x": 10, "y": 64, "z": -20}, decoded[12]
assert decoded[14] == {"dimension": "minecraft:overworld", "x": 8, "y": 70, "z": -16, "angle": 90.0}, decoded[14]
assert decoded[15] == {"none": True} and decoded[16] == {"id": "minecraft:diamond", "count": 3}, decoded[15:17]
assert decoded[17] == {"min": -64, "max": 320}, decoded[17]
assert decoded[18] == {"none": True}, decoded[18]
assert decoded[19] == {**TARGET, "x": -0.5, "y": 100.0, "z": 0.5}, decoded[19]
assert decoded[20] == {"unloaded": True} and decoded[21] is None, decoded[20:22]
assert log == [], log

## The tools of operators

`/golem whoami` tells every sender how Golem sees them. `/golem player`, `world`, `teleport` and
`listname` ask the game, and only the console, a player of operator level 2 and more and another
source of level 2 and more may run them, so a player of level 0 and a source of level 1 are
refused. `player` names where a player is and how they play, `world` the spawn and the heights of a
dimension, `teleport` queues a teleport to the nearest safe place with the rotation the player has,
or says there is none or that its chunks are not loaded, and `listname` queues the name of the player list or its clearing. Unknown
players and dimensions and words that are no numbers answer what is wrong.

In [ ]:
CONSOLE = {"op": "command", "name": "golem"}
AS_ALEX = {**CONSOLE, "senderUuid": ALEX["uuid"], "senderName": ALEX["name"], "level": 2}
AS_SAM = {**CONSOLE, "senderUuid": SAM["uuid"], "senderName": SAM["name"], "level": 0}
BLOCK = {**CONSOLE, "senderName": "@", "level": 2}
LOW = {**CONSOLE, "senderName": "Server", "level": 1}
runs = [
    (CONSOLE, "whoami"), (AS_ALEX, "whoami"), (AS_SAM, "whoami"), (BLOCK, "whoami"),
    (CONSOLE, "player Alex"), (AS_ALEX, "player sam"), (AS_SAM, "player Alex"), (BLOCK, "player Alex"),
    (LOW, "player Alex"), (CONSOLE, "player Nobody"), (CONSOLE, "player"),
    (CONSOLE, "world"), (CONSOLE, "world minecraft:the_nether"), (CONSOLE, "world golem:void"),
    (CONSOLE, "teleport Alex minecraft:overworld 0.2 64 2.9"), (CONSOLE, "teleport Sam minecraft:the_nether 0 64 0"),
    (CONSOLE, "teleport Alex golem:void 0 64 0"), (CONSOLE, "teleport Alex minecraft:overworld x 64 0"),
    (CONSOLE, "teleport Alex minecraft:the_end 0 64 0"),
    (AS_SAM, "teleport Sam minecraft:overworld 0 64 0"),
    (CONSOLE, "listname Alex [AFK] Alex"), (CONSOLE, "listname Alex"), (AS_SAM, "listname Sam x"),
    (CONSOLE, "PLAYER Alex"),
]
steps, log = script("operators", {
    "plugins": [{"kind": "core"}],
    "game": GAME,
    "steps": [{"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"}]
             + [{**sender, "arguments": arguments} for sender, arguments in runs]
             + [{"op": "drain"}],
})
said = {(sender.get("senderName", "console"), arguments): step["answer"] for (sender, arguments), step
        in zip(runs, steps[1:])}
for key, lines in said.items():
    print("%-8s %-45s %s" % (key[0], key[1][:45], " / ".join(lines)[:160]))
refused = ["Only the console and operators of level 2 and more may run /golem %s." % name for name in
           ("player", "teleport", "listname")]
assert said[("console", "whoami")] == ["You run /golem as the console."]
assert said[("Alex", "whoami")] == ["You run /golem as the player Alex at the operator level 2."]
assert said[("Sam", "whoami")] == ["You run /golem as the player Sam at the operator level 0."]
assert said[("@", "whoami")] == ["You run /golem as @ at the level 2."]
alex = said[("console", "player Alex")]
assert alex == [
    "Alex (%s): minecraft:the_nether 1.50 64.00 -2.25, yaw 90.0, pitch -12.5" % ALEX["uuid"],
    "Game mode creative, ping 12 ms, operator level 2, idle 1.5 s",
    "Display name: Alex the Brave",
    "List name: none",
    "Respawn point: minecraft:overworld 10 64 -20",
    "Main hand: 3 minecraft:diamond",
], alex
assert said[("Alex", "player sam")][3:] == ["List name: [AFK] Sam", "Respawn point: none", "Main hand: empty"]
assert said[("Sam", "player Alex")] == [refused[0]] and said[("Server", "player Alex")] == [refused[0]]
assert said[("@", "player Alex")] == alex and said[("console", "PLAYER Alex")] == alex
assert said[("console", "player Nobody")] == ["No player Nobody is online."]
assert said[("console", "player")] == ["Usage: /golem player <name>"]
assert said[("console", "world")] == ["minecraft:overworld: spawn 8 70 -16 at the angle 90.0, blocks from y -64 "
                                      "below y 320"]
assert said[("console", "world minecraft:the_nether")] == ["minecraft:the_nether: spawn 8 70 -16 at the angle "
                                                           "0.0, blocks from y 0 below y 256"]
assert said[("console", "world golem:void")] == ["The server has no dimension golem:void, or the game gives no "
                                                 "answer now."]
assert said[("console", "teleport Alex minecraft:overworld 0.2 64 2.9")] == [
    "Teleported Alex to minecraft:overworld 0.50 65.00 2.50, yaw 90.0, pitch -12.5."]
assert said[("console", "teleport Sam minecraft:the_nether 0 64 0")] == [
    "No safe place within 3 blocks of minecraft:the_nether 0.00 64.00 0.00, yaw -45.0, pitch 0.0, so Sam stays."]
assert said[("console", "teleport Alex golem:void 0 64 0")] == ["The server has no dimension golem:void, or the "
                                                                "game gives no answer now."]
assert said[("console", "teleport Alex minecraft:overworld x 64 0")] == [
    "Usage: /golem teleport <name> <dimension> <x> <y> <z>, with numbers for x, y and z"]
assert said[("console", "teleport Alex minecraft:the_end 0 64 0")] == [
    "The chunks near minecraft:the_end 0.00 64.00 0.00, yaw 90.0, pitch -12.5 are not loaded, so Alex stays, since "
    "the search for a safe place loads no chunk."]
assert said[("Sam", "teleport Sam minecraft:overworld 0 64 0")] == [refused[1]]
assert said[("console", "listname Alex [AFK] Alex")] == ["The player list names Alex [AFK] Alex."]
assert said[("console", "listname Alex")] == ["The player list shows the name of Alex again."]
assert said[("Sam", "listname Sam x")] == [refused[2]]
assert steps[-1]["decoded"] == [
    {"kind": "teleport", "uuid": ALEX["uuid"], "dimension": "minecraft:overworld", "x": 0.5, "y": 65.0, "z": 2.5,
     "yaw": 90.0, "pitch": -12.5},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": "[AFK] Alex"},
    {"kind": "listName", "uuid": ALEX["uuid"], "name": None},
], steps[-1]
assert log == ["the bridge is connected: Forge 47.4.18, Minecraft 1.20.1"], log

## The queue

The queue hands out its actions oldest first, the ones the plugins queue and the ones pushed into
it alike, and a drain empties it. A second install replaces the handler and keeps the queue with
the actions in it.

In [ ]:
pushed = [
    {"kind": "broadcast", "text": "first"},
    {"kind": "command", "command": "say second"},
    {"kind": "message", "uuid": SAM["uuid"], "text": "third"},
]
steps, _ = script("queue", {
    "plugins": [{"kind": "echo", "name": "echo-a"}],
    "steps": [
        {"op": "push", "action": pushed[0]},
        {"op": "playerJoined", **SAM},
        {"op": "push", "action": pushed[1]},
        {"op": "install"},
        {"op": "push", "action": pushed[2]},
        {"op": "commandNames"},
        {"op": "drain"},
        {"op": "drain"},
        {"op": "budgets"},
    ],
})
assert steps[3]["queued"] == 3, "the second install lost the queue: %s" % steps[3]
assert steps[6]["decoded"] == [pushed[0], {"kind": "broadcast", "text": "echo-a playerJoined"}, pushed[1], pushed[2]], steps[6]
assert steps[7]["answer"] == [] and steps[7]["decoded"] == [], "the drain left actions behind: %s" % steps[7]
assert budgets_of(steps)["echo-a"]["calls"] == 0, "the second install kept the old runtime: %s" % budgets_of(steps)
print("drained in order:", [action.get("text") or action.get("command") for action in steps[6]["decoded"]])

## Tick statistics at given instants

`tick-stats` feeds ticks to a fresh window at instants that the cell chooses, so the expected
values are exact. The MSPT is the mean of the durations of the last 100 ticks in milliseconds.
The TPS is the number of intervals between those ticks divided by the time from the first to the
last, at most 20, and 0 while fewer than 2 ticks span no interval.

In [ ]:
def expected(ticks):
    window = ticks[-100:]
    mspt = sum(tick["nanos"] for tick in window) / len(window) / 1e6
    if len(window) < 2:
        return mspt, 0.0
    elapsed = (window[-1]["atMs"] - window[0]["atMs"]) / 1000
    return mspt, 20.0 if elapsed <= 0 else min(20.0, (len(window) - 1) / elapsed)


cases = {
    "one tick": [{"nanos": 7000000, "atMs": 0}],
    "every 100 ms": [{"nanos": 2000000, "atMs": i * 100.0} for i in range(10)],
    "every 25 ms, capped": [{"nanos": 1000000, "atMs": i * 25.0} for i in range(10)],
    "every 50 ms": [{"nanos": 49000000, "atMs": i * 50.0} for i in range(40)],
    "150 irregular ticks": [{"nanos": 1000000 * (i % 7 + 1) + 333 * i, "atMs": i * 50.0 + (i % 5) * 9.5 + (i // 30) * 40.0} for i in range(150)],
    "same instant": [{"nanos": 3000000, "atMs": 10.0}, {"nanos": 5000000, "atMs": 10.0}],
}
for name, ticks in cases.items():
    out, _ = devtool("tick-stats", write("ticks-%d.json" % len(ticks), ticks))
    assert out["window"] == 100 and out["maxTps"] == 20.0, out
    assert out["empty"] == {"ticks": 0, "mspt": 0.0, "tps": 0.0}, out["empty"]
    for count in range(1, len(ticks) + 1):
        mspt, tps = expected(ticks[:count])
        got = out["after"][count - 1]
        assert got["ticks"] == count, (name, count, got)
        assert abs(got["mspt"] - mspt) < 1e-9, (name, count, got, mspt)
        assert abs(got["tps"] - tps) < 1e-6, (name, count, got, tps)
    print("%-22s ticks %3d  mspt %8.4f  tps %7.4f" % (name, len(ticks), out["after"][-1]["mspt"], out["after"][-1]["tps"]))
assert out["after"][-1]["tps"] == 20.0

## Tick statistics on the wall clock

The native `tick` takes the instant of the call itself. The cell feeds ticks with a pause between
them, so the TPS must come out at or below the rate of the pauses and not far under it, then a
burst without pauses, which the cap holds at 20. A negative duration counts as 0.

In [ ]:
pauses = 6
steps, _ = script("wall", {
    "steps": [
        {"op": "stats"},
        {"op": "tick", "nanos": 3000000},
        {"op": "stats"},
        {"op": "tick", "nanos": 3000000, "repeat": pauses - 1, "sleepMs": tick_pause_ms},
        {"op": "stats"},
        {"op": "tick", "nanos": -5, "repeat": 30},
        {"op": "stats"},
    ],
})
before, one, paced, burst = (step["answer"] for step in steps if step["op"] == "stats")
rate = 1000 / tick_pause_ms
print("one tick:", one)
print("paced:", paced, "at most", rate)
print("burst:", burst)
assert before == {"ticks": 0, "mspt": 0.0, "tps": 0.0}, before
assert one == {"ticks": 1, "mspt": 3.0, "tps": 0.0}, one
assert paced["ticks"] == pauses and paced["mspt"] == 3.0, paced
assert 0.8 * rate <= paced["tps"] <= rate + 1e-9, "the paced TPS %.4f is not near %.1f" % (paced["tps"], rate)
assert burst["ticks"] == pauses + 30 and burst["tps"] == 20.0, burst
assert abs(burst["mspt"] - 3.0 * pauses / (pauses + 30)) < 1e-9, burst

## Pass values

With a handler that panics in every method, every native must answer the pass value of the
contract, nothing, false or an empty list, and log the panic, while the queue and the ticks go on.
Without a handler the natives answer the pass value as well. A null where the contract names a
string is logged and answered with the pass value.

In [ ]:
calls = [
    {"op": "serverStarting", "forgeVersion": "47.4.18", "minecraftVersion": "1.20.1"},
    {"op": "serverStarted"},
    {"op": "tick", "nanos": 1000000},
    {"op": "playerJoined", **ALEX},
    {"op": "chat", **ALEX, "message": "spam"},
    {"op": "death", **ALEX, "cause": "minecraft:fall", **NETHER},
    {"op": "respawn", **ALEX, "dimension": "minecraft:overworld"},
    {"op": "playerLeft", **ALEX},
    {"op": "commandNames"},
    {"op": "command", "name": "golem", "arguments": "status"},
    {"op": "permission", **{"uuid": ALEX["uuid"]}, "node": "forge.use_entity_selectors"},
    {"op": "serverStopping"},
    {"op": "serverStopped"},
]
PASS = {"chat": False, "commandNames": [], "command": []}
extra = [{"op": "push", "action": {"kind": "command", "command": "stop"}}, {"op": "drain"}, {"op": "stats"}, {"op": "version"}]
for handler in ["panicking", "none"]:
    steps, log = script("pass-" + handler, {"handler": handler, "steps": calls + extra})
    for step in steps[:len(calls)]:
        assert step["answer"] == PASS.get(step["op"]), (handler, step)
    assert steps[-3]["decoded"] == [{"kind": "command", "command": "stop"}], steps[-3]
    assert steps[-2]["answer"]["ticks"] == 1, steps[-2]
    assert steps[-1]["answer"] == WORKSPACE["version"], steps[-1]
    panicked = [line for line in log if " panicked with " in line]
    print("%-9s pass values %d, panics logged %d" % (handler, len(calls), len(panicked)))
    assert len(panicked) == (len(calls) if handler == "panicking" else 0), log
    assert all(line.startswith("the native ") and line.endswith(", so it answers the pass value") for line in panicked), panicked

steps, log = script("nulls", {
    "plugins": [{"kind": "echo", "name": "echo-a"}],
    "steps": [
        {"op": "chat", "name": "Alex", "message": "spam"},
        {"op": "playerJoined", "uuid": ALEX["uuid"]},
        {"op": "death", **ALEX, **NETHER},
        {"op": "death", **ALEX, "cause": "minecraft:fall"},
        {"op": "serverStarting", "minecraftVersion": "1.20.1"},
        {"op": "drain"},
    ],
})
assert steps[0]["answer"] is False and steps[5]["answer"] == [], steps
assert log == [
    "the native chat got null for uuid, so it answers the pass value",
    "the native playerJoined got null for name, so it answers the pass value",
    "the native death got null for cause, so it answers the pass value",
    "the native death got null for dimension, so it answers the pass value",
    "the native serverStarting got null for forgeVersion, so it answers the pass value",
], log

## Calls from other threads

Forge 47 calls `chat()` on a worker thread while the server thread runs its ticks, so a chat
message can arrive while a plugin still works on a tick. It must wait for the handler and reach
the plugins, not pass unseen. The cell runs a tick whose plugin sleeps, starts two chat messages
on threads of their own while it sleeps, and checks that the censor still cancels the one with
the word spam. A plugin that calls a native from inside the handler is different: that call
would wait for itself, so it answers the pass value at once, and the log names the first such call
of each native.

In [ ]:
hold_ms = 5 * slow_ms
steps, log = script("threads", {
    "budgetMs": budget_ms,
    "plugins": [
        {"kind": "slow", "name": "sluggish", "on": "tick", "sleepMs": hold_ms},
        {"kind": "cancel", "name": "censor", "word": "spam"},
        {"kind": "reenter", "name": "loop", "on": "serverStarted"},
    ],
    "steps": [
        {"op": "together", "staggerMs": slow_ms, "steps": [
            {"op": "tick", "nanos": 1000000},
            {"op": "chat", **ALEX, "message": "buy spam now"},
            {"op": "chat", **SAM, "message": "hello"},
        ]},
        {"op": "serverStarted"},
        {"op": "drain"},
        {"op": "budgets"},
    ],
})
tick, spam, hello = steps[0]["answer"]
for call in (tick, spam, hello):
    print("%-5s answered %-5s after %6.1f ms" % (call["op"], call["answer"], call["endMs"]))
assert tick["endMs"] >= hold_ms, tick
assert spam["answer"] is True, "a chat message beside a running tick escaped the plugins: %s" % spam
assert hello["answer"] is False, hello
assert spam["endMs"] >= 0.95 * hold_ms and hello["endMs"] >= 0.95 * hold_ms, "a chat message did not wait for the tick: %s" % steps[0]
assert broadcasts(steps[2]) == ["loop got 0 names"], steps[2]
calls = {name: row["calls"] for name, row in budgets_of(steps).items()}
print("calls:", calls)
assert calls == {"sluggish": 4, "censor": 4, "loop": 3}, "the cancel of the chat message must end its chain: %s" % calls
for line in log:
    print(line)
assert not any("while the handler runs" in line for line in log), log
assert [line for line in log if "inside the handler" in line] == [
    "the native commandNames was called from inside the handler, so it answers the pass value, and later calls like "
    "it are not logged",
], log
assert len(log) == 2 and log[0].startswith("the plugin sluggish took "), log

## Verdict

The cell clears the temporary directory. A run that reaches it passed every section.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: the 16 natives match okf/bridge.md, and the runtime, the core plugin, the queue the tick statistics and the calls of other threads answer as the wiki says")